In [1]:
# Environment preflight only; this is not the final Task 3 submission.
import json, sys, time, subprocess, importlib.metadata as meta
from pathlib import Path
import torch
out=Path('/kaggle/working/preflight'); out.mkdir(parents=True,exist_ok=True)
env={'python':sys.version,'cuda_available':torch.cuda.is_available(),'torch':torch.__version__,'gpu_count':torch.cuda.device_count(),'devices':[{'name':torch.cuda.get_device_name(i),'vram_bytes':torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]}
(out/'hardware.json').write_text(json.dumps(env,indent=2))
print(env)
assert env['cuda_available'], 'Complete Kaggle phone verification, select GPU T4 x2, and enable Internet first.'
subprocess.run([sys.executable,'-m','pip','install','transformers>=4.51,<5','accelerate','bitsandbytes'],check=True)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID='Qwen/Qwen3-8B'
started=time.monotonic()
tok=AutoTokenizer.from_pretrained(MODEL_ID)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=quant,device_map={'':0},torch_dtype=torch.float16)
info={'model':MODEL_ID,'revision':getattr(model.config,'_commit_hash',None),'load_seconds':time.monotonic()-started,'device_map':str(model.hf_device_map),'memory_allocated_bytes':torch.cuda.memory_allocated(0),'versions':{p:meta.version(p) for p in ['torch','transformers','accelerate','bitsandbytes']}}
(out/'model.json').write_text(json.dumps(info,indent=2))
(out/'requirements-resolved.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(info)
# Synthetic smoke input; none of the 20 supplied records is used here.
messages=[{'role':'user','content':'請用繁體中文寫一句話，說明審核文章需要可核對的證據。'}]
text=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
x=tok(text,return_tensors='pt').to('cuda:0')
started=time.monotonic()
with torch.inference_mode():
    y=model.generate(**x,max_new_tokens=100,do_sample=False,pad_token_id=tok.eos_token_id)
response=tok.decode(y[0,x.input_ids.shape[1]:],skip_special_tokens=True)
assert response.strip(), 'Generation returned no content.'
result={'input':messages,'output':response,'seconds':time.monotonic()-started,'note':'Hardware/model smoke test only; diagnostic quality not yet measured.'}
(out/'generation.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print(response)


{'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'cuda_available': True, 'torch': '2.11.0+cu128', 'gpu_count': 2, 'devices': [{'name': 'Tesla T4', 'vram_bytes': 15636037632}, {'name': 'Tesla T4', 'vram_bytes': 15636037632}]}
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 984.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'model': 'Qwen/Qwen3-8B', 'revision': 'b968826d9c46dd6066d109eabc6255188de91218', 'load_seconds': 134.43370671600002, 'device_map': "{'': 0}", 'memory_allocated_bytes': 6076724736, 'versions': {'torch': '2.11.0+cu128', 'transformers': '4.57.6', 'accelerate': '1.14.0', 'bitsandbytes': '0.50.2'}}
審核文章時，必須提供具體且可核對的證據，以確保資訊的準確性與可靠性。
